# Analytical Solution: Quarter Plate with a Circular Hole
**Author:** Muhammad Mohsin Khan  
**Units:** mm, s, N, tonne; MPa = N/mm²

This notebook calculates stresses and in-plane displacements using the **Kirsch infinite-plate, plane-stress solution** as a reference for the supplied C3D20 model.

**Scope:** these are analytical benchmarks, not an exact solution for the finite 200 × 200 × 10 mm full plate. Finite boundaries and thickness effects can change the FE results.

## Problem Data
- Quarter dimensions: 100 × 100 × 10 mm
- Hole radius: $a=20$ mm
- Young's modulus: $E=210000$ MPa
- Poisson's ratio: $\nu=0.30$
- Tensile traction in $+x$: $\sigma_0=100$ MPa

Polar coordinates are measured from the hole center, with $\theta=0$ along $+x$. The full plate has $d/W=40/200=0.2$; infinite-plate boundary effects are not assumed to vanish.

**Model correction:** the supplied $U_1=0$ on $x=0$ and $U_2=0$ on $y=0$ leave uniform rigid translation in Z free. Add $U_3=0$ at one suitable node. Do not constrain $U_3$ everywhere, which would change the deformation assumption.

In [1]:
import math

Lx = Ly = 100.0   # Quarter dimensions [mm]
t = 10.0         # Thickness [mm]
a = 20.0         # Hole radius [mm]
E = 210000.0     # Young's modulus [MPa]
nu = 0.30
sigma0 = 100.0   # Tensile traction [MPa]
G = E / (2 * (1 + nu))
kappa = (3 - nu) / (1 + nu)  # Plane stress
print(f"Shear modulus = {G:.2f} MPa")
print(f"Quarter loaded-face resultant = {sigma0 * Ly * t:.0f} N")

Shear modulus = 80769.23 MPa
Quarter loaded-face resultant = 100000 N


## 1. Analytical Stress Solution
For $r\geq a$, let $q=(a/r)^2$. The Kirsch stresses are
$$\sigma_{rr}=\frac{\sigma_0}{2}\left[(1-q)+(1-4q+3q^2)\cos 2\theta\right],$$
$$\sigma_{\theta\theta}=\frac{\sigma_0}{2}\left[(1+q)-(1+3q^2)\cos 2\theta\right],$$
$$\tau_{r\theta}=-\frac{\sigma_0}{2}(1+2q-3q^2)\sin 2\theta.$$
At the hole, $\sigma_{rr}=\tau_{r\theta}=0$ and
$$\sigma_{\theta\theta}(a,\theta)=\sigma_0(1-2\cos 2\theta).$$
The maximum is **300 MPa at (0,20) mm**, where hoop stress equals $S_{11}$. At (20,0) mm the hoop stress is **−100 MPa**, corresponding to $S_{22}$.

Source: [Kirsch stresses and finite-width limitations](https://www.fracturemechanics.org/hole.html).

In [2]:
def kirsch_stress(x, y):
    """Return S11, S22, S12, hoop stress [MPa]."""
    r = math.hypot(x, y)
    if r < a - 1e-10:
        raise ValueError("Point lies inside the hole.")
    theta = math.atan2(y, x)
    q = (a / r)**2
    sr = sigma0/2 * (1-q + (1-4*q+3*q*q)*math.cos(2*theta))
    st = sigma0/2 * (1+q - (1+3*q*q)*math.cos(2*theta))
    tr = -sigma0/2 * (1+2*q-3*q*q)*math.sin(2*theta)
    c, s = math.cos(theta), math.sin(theta)
    sx = sr*c*c + st*s*s - 2*tr*s*c
    sy = sr*s*s + st*c*c + 2*tr*s*c
    txy = (sr-st)*s*c + tr*(c*c-s*s)
    return sx, sy, txy, st

for x, y in [(0, a), (a, 0)]:
    sx, sy, txy, hoop = kirsch_stress(x, y)
    print(f"({x:.0f}, {y:.0f}) mm: S11={sx:.2f}, S22={sy:.2f}, hoop={hoop:.2f} MPa")
print("Infinite-plate stress concentration factor: 3.00")

(0, 20) mm: S11=300.00, S22=0.00, hoop=300.00 MPa
(20, 0) mm: S11=0.00, S22=-100.00, hoop=-100.00 MPa
Infinite-plate stress concentration factor: 3.00


## 2. Analytical Displacement Solution
For plane stress,
$$G=\frac{E}{2(1+\nu)},\qquad \kappa=\frac{3-\nu}{1+\nu}.$$
The total displacements, including uniform far-field deformation, are
$$u_r=\frac{\sigma_0}{8G}\left[(\kappa-1)r+\frac{2a^2}{r}+\left(2r+\frac{2(\kappa+1)a^2}{r}-\frac{2a^4}{r^3}\right)\cos 2\theta\right],$$
$$u_\theta=-\frac{\sigma_0}{8G}\left[2r+\frac{2(\kappa-1)a^2}{r}+\frac{2a^4}{r^3}\right]\sin 2\theta.$$
Convert to global components:
$$U_1=u_r\cos\theta-u_\theta\sin\theta,\qquad U_2=u_r\sin\theta+u_\theta\cos\theta.$$
These expressions integrate the Kirsch strains using plane-stress elasticity and satisfy the in-plane symmetry conditions. A numerical strain check is included below.

**There is no single free-end displacement:** the loaded edge displacement varies with position. Compare the same coordinates and components in the FE results, not an arbitrary maximum.

In [3]:
def kirsch_displacement(x, y):
    """Return total plane-stress U1, U2 [mm]."""
    r = math.hypot(x, y)
    if r < a - 1e-10:
        raise ValueError("Point lies inside the hole.")
    theta = math.atan2(y, x)
    p = sigma0 / (8*G)
    ur = p*((kappa-1)*r + 2*a*a/r
        + (2*r + 2*(kappa+1)*a*a/r - 2*a**4/r**3)*math.cos(2*theta))
    ut = -p*(2*r + 2*(kappa-1)*a*a/r + 2*a**4/r**3)*math.sin(2*theta)
    c, s = math.cos(theta), math.sin(theta)
    return ur*c-ut*s, ur*s+ut*c

print(f"{'x [mm]':>8} {'y [mm]':>8} {'U1 [mm]':>12} {'U2 [mm]':>12}")
for x, y in [(a, 0), (0, a), (Lx, 0), (Lx, Ly/2), (Lx, Ly)]:
    ux, uy = kirsch_displacement(x, y)
    print(f"{x:8.1f} {y:8.1f} {ux:12.6f} {uy:12.6f}")

  x [mm]   y [mm]      U1 [mm]      U2 [mm]
    20.0      0.0     0.028571     0.000000
     0.0     20.0     0.000000    -0.009524
   100.0      0.0     0.052617     0.000000
   100.0     50.0     0.050858    -0.006622
   100.0    100.0     0.048917    -0.014346


### Useful Displacement Benchmarks
At the hole boundary:
$$U_1(a,0)=\frac{3\sigma_0a}{E}=0.028571\ \mathrm{mm},$$
$$U_2(0,a)=-\frac{\sigma_0a}{E}=-0.009524\ \mathrm{mm}.$$
The other in-plane component is zero at each of these symmetry points.

At the loaded-edge point (100,0) mm, the infinite-plate reference is $U_1\approx0.052617$ mm.

For comparison only, the **unperforated** uniform-stress plate gives $\sigma_0L_x/E=0.047619$ mm. This neglects the hole and is not the displacement solution of the perforated plate.

## 3. Consistency Checks
Check the hole stresses, symmetry, and displacement-derived strains against plane-stress elasticity. These checks validate the analytical implementation, not the finite-element model.

In [4]:
assert math.isclose(kirsch_stress(0, a)[0], 3*sigma0, abs_tol=1e-9)
assert math.isclose(kirsch_stress(a, 0)[1], -sigma0, abs_tol=1e-9)
assert abs(kirsch_displacement(0, 50)[0]) < 1e-12
assert abs(kirsch_displacement(50, 0)[1]) < 1e-12

# Central-difference strain check at an interior point
x, y, h = 40.0, 35.0, 1e-3
uxp, uyp = kirsch_displacement(x+h, y)
uxm, uym = kirsch_displacement(x-h, y)
vxp, vyp = kirsch_displacement(x, y+h)
vxm, vym = kirsch_displacement(x, y-h)
ex = (uxp-uxm)/(2*h)
ey = (vyp-vym)/(2*h)
gxy = (vxp-vxm+uyp-uym)/(2*h)
sx, sy, txy, _ = kirsch_stress(x, y)
assert math.isclose(ex, (sx-nu*sy)/E, rel_tol=1e-6, abs_tol=1e-10)
assert math.isclose(ey, (sy-nu*sx)/E, rel_tol=1e-6, abs_tol=1e-10)
assert math.isclose(gxy, txy/G, rel_tol=1e-6, abs_tol=1e-10)
print("All stress, symmetry, and strain-displacement checks passed.")

All stress, symmetry, and strain-displacement checks passed.


## Summary
- Maximum infinite-plate hoop / axial stress: **300 MPa** at (0,20) mm; compare $S_{11}$.
- Hole displacement at (20,0) mm: **$U_1=0.028571$ mm**.
- Hole displacement at (0,20) mm: **$U_2=-0.009524$ mm**.
- Loaded-edge displacement at (100,0) mm: **$U_1\approx0.052617$ mm**.

These are **infinite-plate plane-stress references**, not exact predictions for the finite 3D model. Finite boundary and thickness effects can persist after mesh convergence. Compare at identical coordinates and inspect through-thickness variation. Remove the remaining rigid Z translation with a suitable single-node $U_3$ anchor.

### References
1. [MIT: Stresses Around a Circular Hole](https://web.mit.edu/course/3/3.11/www/vg/kirsch.pdf).
2. [Fracture Mechanics: Stress Concentrations at Holes](https://www.fracturemechanics.org/hole.html).
3. [Folias and Wang (1990): Three-dimensional stress field around a circular hole](https://www.math.utah.edu/~folias/1990/Folias1990b.pdf).

No FE data or synthetic FE results are included. The displacement expressions are checked against the Kirsch stress-derived strains in Section 3.